In [34]:
import torch
import torch.nn as nn
import torch.optim as optim


In [35]:
X = torch.randn(1000, 1)
y = 2 * X + 1 + torch.randn(1000, 1) * 0.1

In [36]:
def get_model():
    return nn.Linear(1,1)

loss_func =nn.MSELoss()
epochs = 50

## Batch Gradient Descent
You calculate the error (loss) for the entire dataset at once, average it, and then take a single step to update your weights. It's very stable but requires a lot of memory.

In [37]:
model_batch = get_model()

optimizer_batch = optim.SGD(model_batch.parameters(), lr = 0.1)


In [38]:
for epoch in range(epochs):
    optimizer_batch.zero_grad()
    predictions = model_batch(X)
    loss= loss_func(predictions, y)
    loss.backward()
    optimizer_batch.step()

In [39]:
print(f'Final Loss: {loss.item()}')

Final Loss: 0.009899457916617393


## Stochastic Gradient Descent (SGD)
You calculate the error and update the weights for every single data point, one at a time. It is highly erratic and "noisy," but it uses almost no memory and can escape local minimums easily.

In [40]:
model_sgd = get_model()
optimizer_sgd = optim.SGD(model_sgd.parameters(), lr = 0.1)

In [41]:
for epoch in range(5):
    for i in range(len(X)):
        x_i = X[i:i+1]
        y_i = y[i:i+1]

        optimizer_sgd.zero_grad()
        predictions = model_sgd(x_i)
        loss = loss_func(predictions, y_i)
        loss.backward()
        optimizer_sgd.step()

In [42]:
print(f'Final Loss: {loss.item()}')

Final Loss: 0.012645754031836987


## Mini Batch Gradient Descent
The gold standard of deep learning. You split your data into "batches" (e.g., 32 samples at a time). It offers the stability of Batch GD and the speed/memory efficiency of pure SGD.


In [43]:
from torch.utils.data import TensorDataset, DataLoader
model_mini = get_model()
optimizer_mini = optim.SGD(model_mini.parameters(), lr = 0.1)


In [44]:
dataset = TensorDataset(X, y)
dataloader = DataLoader(dataset, batch_size=32, shuffle = True)

for epoch in range(100):
    for batch_X, batch_y in dataloader:
        optimizer_mini.zero_grad()
        predictions = model_mini(batch_X)
        loss = loss_func(predictions, batch_y)
        loss.backward()
        optimizer_mini.step()

In [45]:
print(f'Final loss: {loss.item()}')

Final loss: 0.016436556354165077


## Second Order method (L-BFGS)
First-order methods (like SGD) only look at the slope (first derivative) to figure out which way is "down." Second-order methods look at the slope and the curvature (second derivative/Hessian matrix). This allows them to take massive, highly accurate steps toward the minimum.

In [46]:
model = get_model()
optimizer = optim.LBFGS(model.parameters(), lr = 0.1)

for epoch in range(10):
    def closure():
        optimizer.zero_grad()
        predictions = model(X)
        loss = loss_func(predictions, y)
        loss.backward()
        return loss
    optimizer.step(closure)
    loss = loss_func(model(X), y)

In [47]:
print(f'Final loss: {loss.item()}')

Final loss: 0.009899458847939968
